In [1]:
#function
# function
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mean_squared_error

def evaluate_deconvolution(pred, true, eps=1e-12):
    """
    评估空间去卷积预测结果与真实值的多种指标。
    
    参数：
        pred : np.ndarray, shape (n_spots, n_celltypes)
            预测的细胞类型比例矩阵。
        true : np.ndarray, shape (n_spots, n_celltypes)
            真实的细胞类型比例矩阵。
        eps : float
            用于避免对数值零的微小常数。
    
    返回：
        pd.DataFrame : 包含所有评估指标的 DataFrame，索引为指标名称。
    """
    # 确保输入为 numpy 数组
    pred = np.asarray(pred)
    true = np.asarray(true)
    assert pred.shape == true.shape, "预测矩阵和真实矩阵形状不一致"
    n_spots, n_types = pred.shape
    
    # 为避免因全零行导致的相关系数计算问题，过滤掉所有值均为常数的行/列
    # 但为简单起见，我们直接计算并处理 nan
    
    # ---- 1. 按 spot 计算 PCC 和 Spearman ----
    spot_pcc = []
    spot_spearman = []
    for i in range(n_spots):
        p = pred[i, :]
        t = true[i, :]
        # 跳过所有值相同的行（无法计算相关系数）
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            spot_pcc.append(pcc)
            spot_spearman.append(spearman)
        # 否则跳过（不计入平均）
    mean_spot_pcc = np.nanmean(spot_pcc) if spot_pcc else np.nan
    mean_spot_spearman = np.nanmean(spot_spearman) if spot_spearman else np.nan
    
    # ---- 2. 按 cell type 计算 PCC 和 Spearman ----
    type_pcc = []
    type_spearman = []
    for j in range(n_types):
        p = pred[:, j]
        t = true[:, j]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            type_pcc.append(pcc)
            type_spearman.append(spearman)
    mean_type_pcc = np.nanmean(type_pcc) if type_pcc else np.nan
    mean_type_spearman = np.nanmean(type_spearman) if type_spearman else np.nan
    
    # ---- 3. 整体 MSE 和 RMSE ----
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    
    # ---- 4. Jensen-Shannon 散度 (按 spot 平均) ----
    # 对每个 spot，归一化预测和真实分布（如果已经为概率则无需，但加保护）
    jsd_list = []
    for i in range(n_spots):
        p = pred[i, :]
        t = true[i, :]
        # 确保非负
        p = np.maximum(p, 0)
        t = np.maximum(t, 0)
        # 归一化为概率分布（和为1）
        p = p / (np.sum(p) + eps)
        t = t / (np.sum(t) + eps)
        # Jensen-Shannon 散度 = (KL(p||m) + KL(t||m)) / 2, m = (p+t)/2
        # scipy 的 jensenshannon 返回 sqrt(JSD)，我们取平方得到 JSD
        jsd = jensenshannon(p, t, base=2) ** 2  # base=2 给出以2为底的对数，结果在[0,1]
        jsd_list.append(jsd)
    mean_jsd = np.mean(jsd_list)
    
    # ---- 组装 DataFrame ----
    results = {
        'metric': [
            'mean_spot_pcc',
            'mean_spot_spearman',
            'mean_type_pcc',
            'mean_type_spearman',
            'overall_mse',
            'overall_rmse',
            'mean_jsd'
        ],
        'value': [
            mean_spot_pcc,
            mean_spot_spearman,
            mean_type_pcc,
            mean_type_spearman,
            mse,
            rmse,
            mean_jsd
        ]
    }
    df = pd.DataFrame(results)
    df.set_index('metric', inplace=True)
    return df



In [ ]:
#ours
import scanpy as sc
import pandas as pd
import numpy as np
import pandas as pd
cell_spot_gt = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Xenium/spots_cells_n.csv',sep=',',header = 0,index_col=0)
import pandas as pd

cell_spot_gt.index='X'+cell_spot_gt.index.astype('str')

spot_clean=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/spatial_coords_clean.tsv',sep='\t',header=0,index_col=0)

In [69]:
spot_clean

,row,col
SpotID,,
0,229.964711,29.184932
1,329.964711,29.184932
2,529.964711,29.184932
3,729.964711,29.184932
4,829.964711,29.184932
...,...,...
2312,4129.964711,5429.184932
2313,6829.964711,5429.184932
2314,7029.964711,5429.184932


In [68]:
cell_spot_gt

,x_coord,y_coord,celltype,spot_id
X12295,7427.360596,1152.664642,DCIS_2,536
X88907,2625.549561,1119.119940,Invasive_Tumor,510
X40802,213.180521,4523.731543,Stromal,2027
X23352,150.475253,1342.689032,Stromal,593
X120092,5503.146533,630.766095,CD4+_T_Cells,300
...,...,...,...,...
X81752,2306.718835,3419.138489,Stromal,1577
X35059,133.333984,4040.037134,Invasive_Tumor,1842
X155036,7032.032568,2046.963525,DCIS_2,934
X112318,4432.914771,704.786963,Unlabeled,337


In [4]:
matrix = pd.get_dummies(cell_spot_gt['spot_id']).astype(int)
A_gt=matrix.copy()
A_gt=A_gt[spot_clean.index]
count_matrix = cell_spot_gt.groupby(['spot_id', 'celltype']).size().unstack(fill_value=0)
count_matrix=count_matrix.div(count_matrix.sum(axis=1), axis=0)
count_matrix=count_matrix.loc[spot_clean.index]
A_gt_deconv=count_matrix.copy()

In [42]:
# novosparc
celltype = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/deconvolution/Cytospace/clean_inputs/cell_type_labels_clean.tsv',sep='\t',header = 0)
novo = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/normalize_novosparc.npy') 


In [6]:
subclass=celltype['CellType'].values
df_onehot = pd.get_dummies(subclass) 
class_names = df_onehot.columns.tolist()
X = df_onehot.values.T 
print(f"矩阵 X 的形状: {X.shape}") # 应输出 (23, 4198)
result = np.dot(X, novo)
result=result.T
row_sums = result.sum(axis=1)  # 每个 spot 的总和，形状 (424,)
spot_ct_proportions = result / row_sums[:, np.newaxis]  # 或者 result / row_sums.reshape(-1,1)
df_results_novosparc = evaluate_deconvolution(spot_ct_proportions, A_gt_deconv.values)

矩阵 X 的形状: (20, 19826)


In [7]:
df_results_novosparc


,value
metric,
mean_spot_pcc,0.595526
mean_spot_spearman,0.356365
mean_type_pcc,0.321788
mean_type_spearman,0.249715
overall_mse,0.023807
overall_rmse,0.154297
mean_jsd,0.481379


In [8]:
import pandas as pd

celltype_list = [
    'Stromal', 'DCIS_2', 'Invasive_Tumor', 'Myoepi_ACTA2',
    'CD8_T_Cells', 'CD4_T_Cells', 'DCIS_1', 'Unlabeled', 'Endothelial',
    'Myoepi_KRT15', 'Macrophages_1', 'B_Cells', 'Perivascular_Like',
    'Prolif_Invasive_Tumor', 'IRF7_DCs', 'Stromal_T_Cell_Hybrid',
    'Macrophages_2', 'T_Cell_Tumor_Hybrid', 'LAMP3_DCs', 'Mast_Cells'
]

tangram_columns = [
    'Stromal', 'DCIS_2', 'Invasive_Tumor', 'Myoepi_ACTA2+', 'CD8+_T_Cells',
    'CD4+_T_Cells', 'DCIS_1', 'Unlabeled', 'Endothelial', 'Myoepi_KRT15+',
    'Macrophages_1', 'B_Cells', 'Perivascular-Like',
    'Prolif_Invasive_Tumor', 'IRF7+_DCs', 'Stromal_&_T_Cell_Hybrid',
    'Macrophages_2', 'T_Cell_&_Tumor_Hybrid', 'LAMP3+_DCs', 'Mast_Cells'
]

mapping_df = pd.DataFrame({
    'celltype': celltype_list,
    'tangram_column': tangram_columns
})

print(mapping_df)

                 celltype           tangram_column
0                 Stromal                  Stromal
1                  DCIS_2                   DCIS_2
2          Invasive_Tumor           Invasive_Tumor
3            Myoepi_ACTA2            Myoepi_ACTA2+
4             CD8_T_Cells             CD8+_T_Cells
5             CD4_T_Cells             CD4+_T_Cells
6                  DCIS_1                   DCIS_1
7               Unlabeled                Unlabeled
8             Endothelial              Endothelial
9            Myoepi_KRT15            Myoepi_KRT15+
10          Macrophages_1            Macrophages_1
11                B_Cells                  B_Cells
12      Perivascular_Like        Perivascular-Like
13  Prolif_Invasive_Tumor    Prolif_Invasive_Tumor
14               IRF7_DCs                IRF7+_DCs
15  Stromal_T_Cell_Hybrid  Stromal_&_T_Cell_Hybrid
16          Macrophages_2            Macrophages_2
17    T_Cell_Tumor_Hybrid    T_Cell_&_Tumor_Hybrid
18              LAMP3_DCs      

In [9]:
mapping_df.index=mapping_df['celltype'].values

In [10]:
# tangram
spatial_with_tangram=sc.read("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/spatial_with_tangram-Simulation.h5ad")
tangram=spatial_with_tangram.obsm["tangram_ct_pred"]
tangram=tangram[mapping_df.loc[class_names]['tangram_column'].values].values
tangram=tangram/tangram.sum(axis=1,keepdims=True)
df_results_tangram = evaluate_deconvolution(tangram, A_gt_deconv.values)


In [18]:
df_results_tangram

,value
metric,
mean_spot_pcc,0.722988
mean_spot_spearman,0.394755
mean_type_pcc,0.578365
mean_type_spearman,0.299853
overall_mse,0.018543
overall_rmse,0.136172
mean_jsd,0.370186


In [13]:
#spotiphy
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/Spotiphy_Result-SimulationData-zz/proportion2-simulationdata.csv',header=0,index_col=0)
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
df_results_spotiphy = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values)

In [17]:
df_results_spotiphy

,value
metric,
mean_spot_pcc,0.534674
mean_spot_spearman,0.323295
mean_type_pcc,0.466091
mean_type_spearman,0.283922
overall_mse,0.028884
overall_rmse,0.169952
mean_jsd,0.446673


In [32]:
# spatialDLWS
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/spatialDWLS/spatialDWLS_dec_zztry.csv',sep=',',index_col=0)
#A_dev_pred=A_dev_pred[A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_spatialDLWS = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values)

In [33]:
df_results_spatialDLWS

,value
metric,
mean_spot_pcc,0.528421
mean_spot_spearman,0.390763
mean_type_pcc,0.445993
mean_type_spearman,0.340090
overall_mse,0.027046
overall_rmse,0.164458
mean_jsd,0.478926


In [43]:
celltype

,CellID,CellType
0,140838,Stromal
1,6331,Stromal
2,160220,DCIS_2
3,12295,DCIS_2
4,88907,Invasive_Tumor
...,...,...
19821,25048,Invasive_Tumor
19822,157950,Stromal
19823,161846,Stromal
19824,53444,Invasive_Tumor


In [35]:
celltype

,CellType
13,DCIS_1
21,DCIS_1
25,Unlabeled
36,Unlabeled
77,DCIS_1
...,...
167745,DCIS_2
167750,Myoepi_KRT15
167756,Myoepi_KRT15
167762,DCIS_2


In [41]:
A_gt_deconv

celltype,B_Cells,CD4+_T_Cells,CD8+_T_Cells,DCIS_1,DCIS_2,Endothelial,IRF7+_DCs,Invasive_Tumor,LAMP3+_DCs,Macrophages_1,Macrophages_2,Mast_Cells,Myoepi_ACTA2+,Myoepi_KRT15+,Perivascular-Like,Prolif_Invasive_Tumor,Stromal,Stromal_&_T_Cell_Hybrid,T_Cell_&_Tumor_Hybrid,Unlabeled
SpotID,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,1.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,1.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.666667,0.00,0.000000,0.0,0.166667,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.166667,0.000000,0.0,0.0,0.0
4,0.000000,0.0,0.0,1.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2312,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0
2313,0.000000,0.0,0.0,0.000000,0.00,0.666667,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.333333,0.0,0.0,0.0
2314,0.000000,0.0,0.0,0.500000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.5,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0


In [34]:
#SIMO
celltype.index=celltype['CellID'].values
import pandas as pd
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/SMIO/Location_map1_zztry.csv', index_col=0)
matrix = df.pivot(index='cell', columns='spot', values='value')
matrix = matrix.fillna(0)

celltype=celltype.loc[matrix.index]
ct_onehot_df = pd.get_dummies(celltype['subclass']) 
spot_ct_proportions = matrix.T @ ct_onehot_df  # n_spots × n_types
row_sums = spot_ct_proportions.sum(axis=1)  # Series
spot_ct_proportions = spot_ct_proportions.div(row_sums, axis=0)  # 按
spot_ct_proportions=spot_ct_proportions[A_gt_deconv.columns]
spot_ct_proportions=spot_ct_proportions.loc[A_gt_deconv.index]
pred = spot_ct_proportions.values.astype(np.float64)
df_results_SIMO = evaluate_deconvolution(pred, A_gt_deconv.values)

KeyError: 'CellID'

In [49]:
import numpy as np
import pandas as pd

# ---- 1. 复制 celltype ----
celltype_use = celltype.copy()
if 'CellID' in celltype_use.columns:
    celltype_use = celltype_use.set_index('CellID')

# ---- 2. 读 df 构造 matrix ----
df = pd.read_csv(
    '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/SMIO/Location_map1_zztry.csv',
    index_col=0,
)
matrix = df.pivot(index='cell', columns='spot', values='value').fillna(0)

# ---- 3. 统一索引类型 ----
matrix.index = matrix.index.astype(str)
celltype_use.index = celltype_use.index.astype(str)

# ---- 4. 取交集，只保留两边都有的 cell ----
common = matrix.index.intersection(celltype_use.index)
matrix = matrix.loc[common]
celltype_use = celltype_use.loc[common]

# ---- 5. 独热编码 ----
ct_onehot_df = pd.get_dummies(celltype_use['CellType'])

# ---- 6. spot × celltype 比例 ----
spot_ct_proportions = matrix.T @ ct_onehot_df
row_sums = spot_ct_proportions.sum(axis=1)
spot_ct_proportions = spot_ct_proportions.div(row_sums.replace(0, 1), axis=0)

# ---- 7. 名称映射：pred 名 -> GT 名 ----
name_map = {
    'B_Cells':                'B_Cells',
    'CD4_T_Cells':            'CD4+_T_Cells',
    'CD8_T_Cells':            'CD8+_T_Cells',
    'DCIS_1':                 'DCIS_1',
    'DCIS_2':                 'DCIS_2',
    'Endothelial':            'Endothelial',
    'IRF7_DCs':               'IRF7+_DCs',
    'Invasive_Tumor':         'Invasive_Tumor',
    'LAMP3_DCs':              'LAMP3+_DCs',
    'Macrophages_1':          'Macrophages_1',
    'Macrophages_2':          'Macrophages_2',
    'Mast_Cells':             'Mast_Cells',
    'Myoepi_ACTA2':           'Myoepi_ACTA2+',
    'Myoepi_KRT15':           'Myoepi_KRT15+',
    'Perivascular_Like':      'Perivascular-Like',
    'Prolif_Invasive_Tumor':  'Prolif_Invasive_Tumor',
    'Stromal':                'Stromal',
    'Stromal_T_Cell_Hybrid':  'Stromal_&_T_Cell_Hybrid',
    'T_Cell_Tumor_Hybrid':    'T_Cell_&_Tumor_Hybrid',
    'Unlabeled':              'Unlabeled',
}

spot_ct_proportions = spot_ct_proportions.rename(columns=name_map)

# ---- 8. 保险：补 0 列 + 按 GT 顺序重排 ----
for col in A_gt_deconv.columns:
    if col not in spot_ct_proportions.columns:
        spot_ct_proportions[col] = 0.0
        print(f"⚠️ 补 0 列: {col}")

spot_ct_proportions = spot_ct_proportions[A_gt_deconv.columns]

# ---- 9. 对齐行（spot）----
missing_spots = set(A_gt_deconv.index) - set(spot_ct_proportions.index)
if missing_spots:
    print(f"GT 中有但 pred 没有的 spot 数: {len(missing_spots)}")
    spot_ct_proportions = spot_ct_proportions.reindex(
        A_gt_deconv.index, fill_value=0.0
    )
else:
    spot_ct_proportions = spot_ct_proportions.loc[A_gt_deconv.index]

# ---- 10. 评估 ----
pred = spot_ct_proportions.values.astype(np.float64)
df_results_SIMO = evaluate_deconvolution(pred, A_gt_deconv.values)
print(df_results_SIMO)

/tmp/ipykernel_3500688/2962527097.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spot_ct_proportions = spot_ct_proportions.div(row_sums.replace(0, 1), axis=0)


                       value
metric                      
mean_spot_pcc       0.769074
mean_spot_spearman  0.712308
mean_type_pcc       0.618637
mean_type_spearman  0.519725
overall_mse         0.014846
overall_rmse        0.121845
mean_jsd            0.239156


In [50]:
print("matrix 原始 cell 数:", len(matrix))
print("celltype 原始 cell 数:", len(celltype_use))
print("交集 cell 数:", len(common))

only_in_matrix = matrix.index.difference(celltype_use.index)
only_in_celltype = celltype_use.index.difference(matrix.index)
print("只在 matrix 里的 cell 数:", len(only_in_matrix))
print("只在 celltype 里的 cell 数:", len(only_in_celltype))

print("\n只在 matrix 里的 cell 示例:", only_in_matrix[:10].tolist())
print("只在 celltype 里的 cell 示例:", only_in_celltype[:10].tolist())

matrix 原始 cell 数: 11489
celltype 原始 cell 数: 11489
交集 cell 数: 11489
只在 matrix 里的 cell 数: 0
只在 celltype 里的 cell 数: 0

只在 matrix 里的 cell 示例: []
只在 celltype 里的 cell 示例: []


In [62]:
df_results_SIMO

,value
metric,
mean_spot_pcc,0.501327
mean_spot_spearman,0.424934
mean_type_pcc,0.402027
mean_type_spearman,0.340523
overall_mse,0.017423
overall_rmse,0.131997
mean_jsd,0.467232


In [51]:
#cytospace
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/deconvolution/Cytospace/cytospace_results_clean/fractional_abundances_by_spot.csv',sep=',',header=0,index_col=0)

In [59]:
A_gt_deconv

celltype,B_Cells,CD4+_T_Cells,CD8+_T_Cells,DCIS_1,DCIS_2,Endothelial,IRF7+_DCs,Invasive_Tumor,LAMP3+_DCs,Macrophages_1,Macrophages_2,Mast_Cells,Myoepi_ACTA2+,Myoepi_KRT15+,Perivascular-Like,Prolif_Invasive_Tumor,Stromal,Stromal_&_T_Cell_Hybrid,T_Cell_&_Tumor_Hybrid,Unlabeled
SpotID,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,1.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,1.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.666667,0.00,0.000000,0.0,0.166667,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.166667,0.000000,0.0,0.0,0.0
4,0.000000,0.0,0.0,1.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2312,0.000000,0.0,0.0,0.000000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0
2313,0.000000,0.0,0.0,0.000000,0.00,0.666667,0.0,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.000000,0.333333,0.0,0.0,0.0
2314,0.000000,0.0,0.0,0.500000,0.00,0.000000,0.0,0.000000,0.0,0.00,0.0,0.0,0.5,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0


In [57]:
A_dev_pred

,DCIS_2,Myoepi_ACTA2,CD4_T_Cells,Unlabeled,Myoepi_KRT15,B_Cells,CD8_T_Cells,Invasive_Tumor,Macrophages_1,Perivascular_Like,DCIS_1,Stromal,Prolif_Invasive_Tumor,Endothelial,IRF7_DCs,Stromal_T_Cell_Hybrid,Macrophages_2,T_Cell_Tumor_Hybrid,LAMP3_DCs,Mast_Cells
SpotID,,,,,,,,,,,,,,,,,,,,
1343,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1185,0.60,0.0,0.0,0.0,0.2,0.0,0.0,0.00,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1745,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1907,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1657,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2310,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1682,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2272,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [61]:
import numpy as np
import pandas as pd

# ---- 1. 复制一份，别改原始对象 ----
A_dev_pred_aligned = A_dev_pred.copy()

# ---- 2. 列名映射：pred 名 -> GT 名 ----
name_map = {
    'B_Cells':                'B_Cells',
    'CD4_T_Cells':            'CD4+_T_Cells',
    'CD8_T_Cells':            'CD8+_T_Cells',
    'DCIS_1':                 'DCIS_1',
    'DCIS_2':                 'DCIS_2',
    'Endothelial':            'Endothelial',
    'IRF7_DCs':               'IRF7+_DCs',
    'Invasive_Tumor':         'Invasive_Tumor',
    'LAMP3_DCs':              'LAMP3+_DCs',
    'Macrophages_1':          'Macrophages_1',
    'Macrophages_2':          'Macrophages_2',
    'Mast_Cells':             'Mast_Cells',
    'Myoepi_ACTA2':           'Myoepi_ACTA2+',
    'Myoepi_KRT15':           'Myoepi_KRT15+',
    'Perivascular_Like':      'Perivascular-Like',
    'Prolif_Invasive_Tumor':  'Prolif_Invasive_Tumor',
    'Stromal':                'Stromal',
    'Stromal_T_Cell_Hybrid':  'Stromal_&_T_Cell_Hybrid',
    'T_Cell_Tumor_Hybrid':    'T_Cell_&_Tumor_Hybrid',
    'Unlabeled':              'Unlabeled',
}

A_dev_pred_aligned = A_dev_pred_aligned.rename(columns=name_map)

# ---- 3. 补 0 列：GT 有但 pred 没有的类型 ----
for col in A_gt_deconv.columns:
    if col not in A_dev_pred_aligned.columns:
        A_dev_pred_aligned[col] = 0.0
        print(f"⚠️ 补 0 列: {col}")

# ---- 4. 按 GT 列顺序重排 ----
A_dev_pred_aligned = A_dev_pred_aligned[A_gt_deconv.columns]

# ---- 5. 行对齐：GT 的 spot 为准 ----
missing_spots = set(A_gt_deconv.index) - set(A_dev_pred_aligned.index)
extra_spots   = set(A_dev_pred_aligned.index) - set(A_gt_deconv.index)

print(f"GT spot 数:      {len(A_gt_deconv)}")
print(f"pred spot 数:    {len(A_dev_pred_aligned)}")
print(f"GT 有 pred 没有: {len(missing_spots)}")
print(f"pred 有 GT 没有: {len(extra_spots)}")

# 按 GT 的 spot 顺序重排，缺失的 spot 填 0
A_dev_pred_aligned = A_dev_pred_aligned.reindex(
    A_gt_deconv.index, fill_value=0.0
)

# ---- 6. 转 numpy 评估 ----
pred = A_dev_pred_aligned.values.astype(np.float64)
gt   = A_gt_deconv.values.astype(np.float64)

df_results_cytospace = evaluate_deconvolution(pred, gt)
print(df_results_cytospace)

GT spot 数:      2300
pred spot 数:    2288
GT 有 pred 没有: 12
pred 有 GT 没有: 0
                       value
metric                      
mean_spot_pcc       0.607960
mean_spot_spearman  0.585422
mean_type_pcc       0.529342
mean_type_spearman  0.445419
overall_mse         0.025084
overall_rmse        0.158378
mean_jsd                 NaN


/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/scipy/spatial/distance.py:1280: RuntimeWarning: invalid value encountered in divide
  p = p / np.sum(p, axis=axis, keepdims=True)


In [62]:
df_results_cytospace

,value
metric,
mean_spot_pcc,0.607960
mean_spot_spearman,0.585422
mean_type_pcc,0.529342
mean_type_spearman,0.445419
overall_mse,0.025084
overall_rmse,0.158378
mean_jsd,NaN


## cell to spot mapping 

In [63]:
import numpy as np

def get_entropy_uniformity(matrix):
    # 防止 log(0) 报错，加上极小值
    epsilon = 1e-12
    matrix = np.clip(matrix, epsilon, 1)
    
    N = matrix.shape[0] # 行数
    # 计算每列的熵
    entropy = -np.sum(matrix * np.log(matrix), axis=0)
    # 归一化 (除以最大熵 log(N))
    max_entropy = np.log(N)
    uniformity_score = entropy / max_entropy
    
    return uniformity_score


In [64]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_mapping(pred_probs, gt_matrix, spot_coords, k_list=[1,3,5,10]):
    """
    评估 cell-to-spot mapping 的指标。
    
    参数:
        pred_probs: (n_cells, n_spots) 预测概率矩阵
        gt_matrix: (n_cells, n_spots) ground truth 二值矩阵，每行一个1
        spot_coords: (n_spots, 2) spot 坐标
        k_list: list of int, Top-k 准确率的 k 值
    
    返回:
        metrics: dict
    """
    # 确保是 numpy 数组
    pred_probs = np.asarray(pred_probs)
    gt_matrix = np.asarray(gt_matrix)
    spot_coords = np.asarray(spot_coords)
    
    n_cells, n_spots = pred_probs.shape
    assert gt_matrix.shape == (n_cells, n_spots), "形状不匹配"
    assert spot_coords.shape[0] == n_spots, "spot 坐标数量不匹配"
    
    # 真实标签：每行的 argmax，如果全零则忽略
    true_labels = np.argmax(gt_matrix, axis=1)
    valid_mask = gt_matrix.sum(axis=1) > 0
    if not np.all(valid_mask):
        print(f"警告：{np.sum(~valid_mask)} 个细胞在 ground truth 中无标签，已忽略。")
    true_labels = true_labels[valid_mask]
    pred_probs_valid = pred_probs[valid_mask]
    
    n_valid = len(true_labels)
    if n_valid == 0:
        return {}
    
    # 预测标签
    pred_labels = np.argmax(pred_probs_valid, axis=1)
    
    # 1. Mapping accuracy
    mapping_acc = np.mean(pred_labels == true_labels)
    
    # 2. Top-k accuracy
    topk_acc = {}
    for k in k_list:
        if k > n_spots:
            k = n_spots
        # 获取每个细胞预测概率最高的 k 个 spot 索引
        topk_indices = np.argsort(-pred_probs_valid, axis=1)[:, :k]
        # 检查真实标签是否在其中
        correct = np.any(topk_indices == true_labels[:, None], axis=1)
        topk_acc[f'top{k}_acc'] = np.mean(correct)
    
    # 3. AUROC / AUPRC (宏平均)
    auroc_list = []
    auprc_list = []
    for j in range(n_spots):
        y_true = (true_labels == j).astype(int)
        if y_true.sum() == 0:
            continue  # 没有正样本，跳过
        y_score = pred_probs_valid[:, j]
        try:
            auroc_list.append(roc_auc_score(y_true, y_score))
            auprc_list.append(average_precision_score(y_true, y_score))
        except ValueError:
            continue
    macro_auroc = np.mean(auroc_list) if auroc_list else np.nan
    macro_auprc = np.mean(auprc_list) if auprc_list else np.nan
    
    # 4. 坐标 MAE / RMSE
    # 真实坐标
    true_coords = spot_coords[true_labels]
    # 预测坐标（软分配：加权平均）
    # 归一化预测概率（确保每行和为1）
    pred_probs_norm = pred_probs_valid / (pred_probs_valid.sum(axis=1, keepdims=True) + 1e-12)
    pred_coords = pred_probs_norm @ spot_coords  # (n_valid, 2)
    # 欧氏距离
    dists = np.linalg.norm(true_coords - pred_coords, axis=1)
    coord_mae = np.mean(dists)
    coord_rmse = np.sqrt(np.mean(dists**2))
    
    # 汇总
    metrics = {
        'mapping_accuracy': mapping_acc,
        'coord_mae': coord_mae,
        'coord_rmse': coord_rmse,
        'macro_auroc': macro_auroc,
        'macro_auprc': macro_auprc,
    }
    metrics.update(topk_acc)
    return metrics

In [65]:
pred_A=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/Ours/predicted_A_matrix.csv',sep=',',header=0,index_col=0)

In [66]:
spatial_coords=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/spatial_coords.tsv', sep='\t')

In [7]:
A_gt=np.load('/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/groundtruth_A_m1s1s50.npy')

In [8]:
assigned_cells = pd.read_csv("/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/spots_cells.csv", header=0)
df_result = pd.crosstab(assigned_cells['spot_id'], assigned_cells['cell_id'])
spots_meta_df=pd.read_csv("/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/spots_info.csv", header=0)
df_result1 = pd.DataFrame(np.zeros((spots_meta_df.shape[0],A_gt.shape[1])), index=spots_meta_df['spot_id'], columns=pred_A.columns)
df_result1[df_result.columns]= df_result.values
df_result1.index=df_result.index


In [9]:
metrics_ours=evaluate_mapping(pred_A.T,df_result1.T,spatial_coords[['row','col']].values)

警告：869 个细胞在 ground truth 中无标签，已忽略。


In [10]:
uniformity_score=get_entropy_uniformity(pred_A)
import numpy as np

# 假设 uniformity_score 是形状 (4198,) 的数组
top_n = 2120
top_indices = np.argsort(-uniformity_score)[-top_n:]

In [11]:
metrics_ours1=evaluate_mapping(pred_A.T.iloc[top_indices.values,:],df_result1.T.iloc[top_indices.values,:],spatial_coords[['row','col']].values) 

警告：445 个细胞在 ground truth 中无标签，已忽略。


In [138]:
metrics_ours1

{'mapping_accuracy': np.float64(0.10149253731343283),
 'coord_mae': np.float64(657.0490031540352),
 'coord_rmse': np.float64(783.1301213889872),
 'macro_auroc': np.float64(0.7873712223209481),
 'macro_auprc': np.float64(0.09953259546158931),
 'top1_acc': np.float64(0.10149253731343283),
 'top3_acc': np.float64(0.15940298507462686),
 'top5_acc': np.float64(0.21014925373134327),
 'top10_acc': np.float64(0.2829850746268657)}

In [13]:
#novosparc
novo = np.load('/home/qyyuan/project/ST_CP/JE/compare/novosparc/A_SimulationData-alpha0.5-epsilon-3-2026-04-22.npy') 

metrics_novo=evaluate_mapping(novo,df_result1.T,spatial_coords[['row','col']].values)

警告：869 个细胞在 ground truth 中无标签，已忽略。


In [92]:
metrics_novo

{'mapping_accuracy': np.float64(0.033343346350255335),
 'coord_mae': np.float64(660.7084757282876),
 'coord_rmse': np.float64(722.7369913382175),
 'macro_auroc': np.float64(0.7676185011524929),
 'macro_auprc': np.float64(0.02538794773402823),
 'top1_acc': np.float64(0.033343346350255335),
 'top3_acc': np.float64(0.08110543706818865),
 'top5_acc': np.float64(0.11685190747972364),
 'top10_acc': np.float64(0.185641333733854)}

In [15]:
#tangram
import scanpy as sc
mapping_tangram=sc.read('/home/qyyuan/project/ST_CP/JE/Tangram/tangram_mapping-Simulation.h5ad')
metrics_tangram=evaluate_mapping(mapping_tangram.X,df_result1.T,spatial_coords[['row','col']].values)

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


警告：869 个细胞在 ground truth 中无标签，已忽略。


In [102]:
metrics_tangram

{'mapping_accuracy': np.float64(0.07870231300690898),
 'coord_mae': np.float64(686.5221262849681),
 'coord_rmse': np.float64(797.8499277338441),
 'macro_auroc': np.float64(0.740349929886425),
 'macro_auprc': np.float64(0.06336020473972773),
 'top1_acc': np.float64(0.07870231300690898),
 'top3_acc': np.float64(0.14268549113848003),
 'top5_acc': np.float64(0.1820366476419345),
 'top10_acc': np.float64(0.2529288074496846)}

In [16]:
#SIMO
import pandas as pd
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/SMIO/Location_map1_zztry.csv', index_col=0)
matrix = df.pivot(index='cell', columns='spot', values='value')
matrix = matrix.fillna(0)
metrics_SIMO=evaluate_mapping(matrix,df_result1.T.loc[matrix.index],spatial_coords[['row','col']].values)

警告：428 个细胞在 ground truth 中无标签，已忽略。


In [128]:
matrix.shape

(2120, 424)

In [17]:
metrics_SIMO

{'mapping_accuracy': np.float64(0.10815602836879433),
 'coord_mae': np.float64(782.7659295393025),
 'coord_rmse': np.float64(973.6329007483716),
 'macro_auroc': np.float64(0.5652910476248295),
 'macro_auprc': np.float64(0.05003857451142495),
 'top1_acc': np.float64(0.10815602836879433),
 'top3_acc': np.float64(0.11052009456264776),
 'top5_acc': np.float64(0.11170212765957446),
 'top10_acc': np.float64(0.1235224586288416)}

In [18]:
#cytospace
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/cytospace_results/assigned_locations.csv')

# 创建全0矩阵，行是 OriginalCID，列是 SpotID
# 方法1：使用 crosstab
matrix = pd.crosstab(df['OriginalCID'], df['SpotID'])
# 这会自动给每个组合计数，由于每个组合只有一个，所以值会是1。
# 但是 crosstab 只会包含出现过的 SpotID 列。如果需要包含所有 0-423 列，可以 reindex。
# 另外，crosstab 的结果是计数，如果某个组合出现多次，值可能大于1。但这里每个组合应该只出现一次。

# 如果需要确保只有 0/1，可以转换为布尔或 clip
matrix = (matrix > 0).astype(int)

# 如果需要包含所有 0 到 423 的列
all_spot_ids = range(424)  # 假设最大 spot_id 是 423
matrix = matrix.reindex(columns=all_spot_ids, fill_value=0)

# 如果需要确保行顺序与某个 adata_sc.obs_names 一致，可以 reindex 行
# 例如 adata_sc.obs_names 包含这些 OriginalCID
# matrix = matrix.reindex(adata_sc.obs_names, fill_value=0)

In [19]:
metrics_cytoscape=evaluate_mapping(matrix,df_result1.T.loc[matrix.index],spatial_coords[['row','col']].values)

警告：373 个细胞在 ground truth 中无标签，已忽略。


In [161]:
metrics_cytoscape

{'mapping_accuracy': np.float64(0.03706111833550065),
 'coord_mae': np.float64(881.6093603521788),
 'coord_rmse': np.float64(1024.8406305468134),
 'macro_auroc': np.float64(0.5278538990735308),
 'macro_auprc': np.float64(0.027522773791462837),
 'top1_acc': np.float64(0.03706111833550065),
 'top3_acc': np.float64(0.0377113133940182),
 'top5_acc': np.float64(0.03836150845253576),
 'top10_acc': np.float64(0.05071521456436931)}

In [20]:
# CellTrek
matrix=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CellTreck_assignment.txt',sep=' ',header=0,index_col=0)

In [21]:
metrics_CellTrek=evaluate_mapping(matrix,df_result1.T.loc[matrix.index.str[1:]],spatial_coords[['row','col']].values)

警告：387 个细胞在 ground truth 中无标签，已忽略。


In [22]:
metrics_CellTrek

{'mapping_accuracy': np.float64(0.029081295439524125),
 'coord_mae': np.float64(796.1169338849173),
 'coord_rmse': np.float64(971.006291535188),
 'macro_auroc': np.float64(0.5136401368675326),
 'macro_auprc': np.float64(0.009768522403803782),
 'top1_acc': np.float64(0.029081295439524125),
 'top3_acc': np.float64(0.031064111037673495),
 'top5_acc': np.float64(0.031725049570389956),
 'top10_acc': np.float64(0.04560475875743556)}

In [ ]:
# CMAP
matrix=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP/cell_to_spot.txt",sep=' ')
df_result1.index = df_result1.index.astype(str)

In [39]:
df_result2 = df_result1.loc[matrix.columns,]

In [48]:

metrics_CMAP=evaluate_mapping(matrix,df_result2.T.loc[matrix.index],spatial_coords.loc[matrix.columns][['row','col']].values)

警告：494 个细胞在 ground truth 中无标签，已忽略。


In [49]:
metrics_CMAP

{'mapping_accuracy': np.float64(0.026507620941020542),
 'coord_mae': np.float64(919.1767617585247),
 'coord_rmse': np.float64(1062.78256889667),
 'macro_auroc': np.float64(0.5126812314295757),
 'macro_auprc': np.float64(0.01701265173846353),
 'top1_acc': np.float64(0.026507620941020542),
 'top3_acc': np.float64(0.032471835652750164),
 'top5_acc': np.float64(0.03512259774685222),
 'top10_acc': np.float64(0.058979456593770706)}